In [ ]:
!pip install -r ../requirements.txt

In [2]:
# Import das bibliotecas
from pandasql import sqldf
import pandas as pd

In [3]:
# Leitura das bases de dados
df_videos_metadata = pd.read_csv("../data/videos_metadata.csv")
df_titles_genres = pd.read_csv("../data/titles_genres.csv")

In [4]:
# Amostras dos metadados de vídeos
df_videos_metadata.head()

,video_id,created_at,duration,category,title_id
0,11513808,2023-04-06T14:13:34-03:00,1951950.0,Esportes,Y1fdtMxPxz
1,11345132,2023-02-06T18:58:49-03:00,4716879.0,Esportes,dNLC8vDkhF
2,9238620,2021-02-04T10:11:07-03:00,648515.0,Esportes,ffn8n668ps
3,12291782,2024-01-23T20:10:49-03:00,1287919.0,Esportes,R8xSSLDSC2
4,12097480,2023-11-08T15:12:23-03:00,2612543.0,Esportes,qXV7YfpCCp


In [5]:
# Amostras dos gêneros dos títulos
df_titles_genres.head()

,title_id,genres
0,wzB5bzGq7s,Crime
1,V6DKRHLkXn,Crime
2,hRT5rbtQJH,Drama
3,fHY5gN69vc,Drama
4,JMwJjMy59j,Drama


In [6]:
# Exemplo de como executar uma query usando o pandasql
query = '''

SELECT * FROM df_videos_metadata LIMIT 10

'''

In [7]:
# Execução da query anterior
query_result = sqldf(query)
query_result

,video_id,created_at,duration,category,title_id
0,11513808,2023-04-06T14:13:34-03:00,1951950.0,Esportes,Y1fdtMxPxz
1,11345132,2023-02-06T18:58:49-03:00,4716879.0,Esportes,dNLC8vDkhF
2,9238620,2021-02-04T10:11:07-03:00,648515.0,Esportes,ffn8n668ps
3,12291782,2024-01-23T20:10:49-03:00,1287919.0,Esportes,R8xSSLDSC2
4,12097480,2023-11-08T15:12:23-03:00,2612543.0,Esportes,qXV7YfpCCp
5,10689745,2022-06-22T05:58:17-03:00,802468.0,Esportes,FQSrskGh7F
6,12308807,2024-01-29T21:07:41-03:00,225327.0,Esportes,mhChyYSK99
7,12135180,2023-11-22T14:18:42-03:00,6146974.0,Esportes,kNRfLngrvb
8,10225675,2022-01-20T10:26:20-03:00,3220284.0,Esportes,JTP8W79x5c
9,1143193,2009-10-17T14:18:19-03:00,98200.0,Esportes,Y6dFKszwxm


In [8]:
# Questão 1: Faça um join entre as bases (df_videos_metadata e df_titles_genres) pelo id do título de tal forma que a base de metadados de vídeo passe a conter os genêros dos filmes
# LEFT JOIN preserva todos os vídeos: só 835 dos 1.501 têm gênero (títulos que não são filmes ficam com genres = NULL).
# title_id é único em df_titles_genres, então o join não duplica linhas (1.501 linhas de saída).

query_q1 = '''

SELECT
    v.video_id,
    v.created_at,
    v.duration,
    v.category,
    v.title_id,
    g.genres
FROM df_videos_metadata AS v
LEFT JOIN df_titles_genres AS g
    ON v.title_id = g.title_id

'''

In [9]:
query_q1_result = sqldf(query_q1)
query_q1_result

,video_id,created_at,duration,category,title_id,genres
0,11513808,2023-04-06T14:13:34-03:00,1951950.0,Esportes,Y1fdtMxPxz,NaN
1,11345132,2023-02-06T18:58:49-03:00,4716879.0,Esportes,dNLC8vDkhF,NaN
2,9238620,2021-02-04T10:11:07-03:00,648515.0,Esportes,ffn8n668ps,NaN
3,12291782,2024-01-23T20:10:49-03:00,1287919.0,Esportes,R8xSSLDSC2,NaN
4,12097480,2023-11-08T15:12:23-03:00,2612543.0,Esportes,qXV7YfpCCp,Esporte
...,...,...,...,...,...,...
1496,7811218,2019-08-01T16:03:41-03:00,20020.0,Entretenimento,2SN7P9jdQf,NaN
1497,8163268,2019-12-13T13:23:27-03:00,20020.0,Entretenimento,9v6wjFqfY9,NaN
1498,8214629,2020-01-06T12:16:33-03:00,20020.0,Entretenimento,XdtXg6y9B7,NaN
1499,8022637,2019-10-21T20:17:21-03:00,20020.0,Entretenimento,gJFC7QZKRL,NaN


In [10]:
# Questão 2: Faça uma query que calcule o total de vídeos distintos em cada categotia a partir da base query_q1_result e retorne os valores em ordem decrescente

query_q2 = '''

SELECT
    category,
    COUNT(DISTINCT video_id) AS total_videos_distintos
FROM query_q1_result
GROUP BY category
ORDER BY total_videos_distintos DESC

'''

In [11]:
query_q2_result = sqldf(query_q2)
print(query_q2_result)

         category  total_videos_distintos
0  Entretenimento                    1205
1   Sem Categoria                     204
2        Notícias                      49
3        Esportes                      43


In [12]:
# Questão 3: Faça uma query que calcule a média de duração dos títulos em horas para o gênero "Drama" a partir da base query_q1_result. Retorne apenas o gênero "Drama".
# OBS.1: O campo de genres contém uma string que pode ter mais de um gênero ao mesmo tempo
# OBS.2: Não é possível usar regexp aqui, apenas o LIKE
#
# Truque: envolver o campo com o separador ('| ' + genres + '|') faz todo gênero aparecer como '| Drama|',
# esteja ele no início, meio ou fim. Um único LIKE cobre todos os casos e não casa com gêneros que
# apenas contêm a palavra (ex.: um hipotético 'Docudrama').
# duration está em milissegundos -> / 3.600.000 = horas. AVG ignora durações nulas.

query_q3 = '''

SELECT
    'Drama'                         AS genero,
    AVG(duration) / 3600000.0       AS media_duracao_horas
FROM query_q1_result
WHERE '| ' || genres || '|' LIKE '%| Drama|%'

'''

In [13]:
query_q3_result = sqldf(query_q3)
print(query_q3_result)

  genero  media_duracao_horas
0  Drama              1.71864
